# Practice #5. "Machine Learning for Time Series Forecasting"

Feature engineering **without leaking the target**, then Ridge, Lasso,
polynomial features and SVR, compared properly.

Fill in the cells tagged `graded`, keeping every name and signature exactly as
given — they are graded automatically.

One of the grading tests builds features from pure white noise and fails if any
model can predict it. That is the leakage check, and it is the point of this
notebook. Section 1.1 shows you how to run the same check yourself.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.svm import SVR

def find_data_dir():
    """The repo's data/ directory, wherever the kernel happens to start."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "data" / "airline-passengers.csv").exists():
            return folder / "data"
    raise FileNotFoundError("data/ not found - run this notebook inside the repo")


DATA_DIR = globals().get("DATA_DIR", find_data_dir())

## 0. Data

`opsd_germany_daily.csv`, column `Consumption` — German daily electricity demand,
2006-2017. Calendar features earn their place here: weekend demand is visibly
lower than weekday demand.

In [ ]:
def load_series(path, time_col, value_col):
    """Read a CSV into a float Series named "y" on a DatetimeIndex."""
    # TODO: same contract as Practices 1-4.
    raise NotImplementedError

In [ ]:
series = load_series(DATA_DIR / "opsd_germany_daily.csv", "Date", "Consumption")
print(f"{len(series)} points, {series.index.min():%Y-%m-%d} to {series.index.max():%Y-%m-%d}")

fig, axes = plt.subplots(1, 2, figsize=(20, 4))
axes[0].plot(series)
axes[0].set_title("Daily electricity consumption")
series.groupby(series.index.dayofweek).mean().plot.bar(ax=axes[1])
axes[1].set_title("Mean by day of week (0 = Monday)")
plt.tight_layout();

## 1. Feature engineering

A model needs a feature matrix `X` and a target `y`. For a time series you build
`X` out of the series' own past.

### 1.1 The rule this notebook is built around

> **A feature for time `t` may use only information from before `t`.**

Break it and nothing errors — your scores just get *better*. The classic version:

```python
features["rolling_mean_3"] = series.rolling(3).mean()   # WRONG: includes y[t]
features["diff_1"] = series.diff()                      # WRONG: is y[t] - y[t-1]
features["target"] = series
```

That second line makes `target = lag_1 + diff_1` *exactly*, so any linear model
reconstructs the target and reports **R² = 1.000, RMSE = 0.000**. Every model
then scores perfectly, section 4 compares nothing, and you learn nothing.

Fix: build rolling and diff features from `series.shift(1)`, so the newest value
they can see is `y[t-1]`.

Check it on any feature matrix — change `y[t]`, and nothing in row `t` of `X`
may move.

In [ ]:
def make_time_features(index):
    """Calendar features for a DatetimeIndex, as a DataFrame on that index.

    Columns, in order: year, month, day, dayofweek, dayofyear, month_sin,
    month_cos, dayofweek_sin, dayofweek_cos, trend.

    The sin/cos pairs wrap the cycle, so December sits next to January.
    `trend` is 0, 1, 2, ...

    These read the timestamp, never y, so they cannot leak.
    """
    # TODO
    raise NotImplementedError

In [ ]:
def make_lag_features(series, n_lags):
    """Columns lag_1 .. lag_n_lags; lag_k at time t is y[t-k]."""
    # TODO
    raise NotImplementedError


def make_rolling_features(series, windows):
    """rolling_{mean,std,min,max}_w for each w, over PAST values only.

    A plain series.rolling(w) at time t includes y[t] — the value you are
    predicting. The window must end at t-1. See section 1.1.
    """
    # TODO
    raise NotImplementedError


def make_diff_features(series, periods):
    """diff_p and pct_change_p for each p, over PAST values only.

    At time t these may use nothing newer than y[t-1].
    """
    # TODO
    raise NotImplementedError

### 1.2 Assembling the supervised problem

In [ ]:
def build_supervised(series, n_lags=7, windows=(7, 14, 28), diff_periods=(1, 7)):
    """Turn a series into a supervised (X, y) pair sharing one index.

    Concatenate the time, lag, rolling and diff features, attach y[t] as the
    target, then drop every row still holding a NaN. pct_change can produce
    infinities — turn those into NaN first so their rows go too.
    """
    # TODO
    raise NotImplementedError


def split_supervised(X, y, test_size):
    """Return (X_train, X_test, y_train, y_test); last `test_size` rows test."""
    # TODO
    raise NotImplementedError

In [ ]:
X, y = build_supervised(series)
print(f"feature matrix: {X.shape[0]} rows x {X.shape[1]} columns")
print(f"columns: {list(X.columns)}")

# The leakage check, by hand: bump one target value and see what moves.
tampered = series.copy()
moment = series.index[1000]
tampered.loc[moment] += 10_000
X_tampered, _ = build_supervised(tampered)

changed = (X.loc[moment] != X_tampered.loc[moment]).sum()
print(f"\nfeatures at {moment:%Y-%m-%d} that moved when y at that "
      f"same timestamp changed: {changed}")
assert changed == 0, "a feature is reading the present — find it and shift it"

### 1.3 Train / test split

In [ ]:
X_train, X_test, y_train, y_test = split_supervised(X, y, test_size=365)
print(f"train {X_train.shape} up to {X_train.index.max():%Y-%m-%d}")
print(f"test  {X_test.shape} from {X_test.index.min():%Y-%m-%d}")

## 2. Linear models

$$y_t = \beta_0 + \beta_1 x_{1,t} + \ldots + \beta_n x_{n,t} + \epsilon_t$$

Ridge, Lasso and SVR all need scaled inputs — and the scaling must happen
*inside* the fit, so the test set never contributes to the mean and variance.

In [ ]:
def fit_predict(estimator, X_train, y_train, X_test):
    """Scale, fit `estimator`, return predictions for X_test as a Series.

    Put the scaler and the estimator in one Pipeline. Not a style preference:
    a scaler fitted on the whole dataset leaks the test set's mean and variance
    into training. A pipeline fits it on X_train only.
    """
    # TODO
    raise NotImplementedError


def evaluate(y_true, y_pred):
    """Return {"rmse": ..., "mae": ..., "r2": ...} as floats."""
    # TODO
    raise NotImplementedError

In [ ]:
# Keep every prediction, not just its score -- section 4 plots the winner.
predictions = {"naive (y[t-1])": y_test.shift(1).fillna(y_train.iloc[-1])}

for name, estimator in (("LinearRegression", LinearRegression()),
                        ("Ridge(alpha=1)", Ridge(alpha=1.0)),
                        ("Lasso(alpha=0.1)", Lasso(alpha=0.1, max_iter=5000))):
    predictions[name] = fit_predict(estimator, X_train, y_train, X_test)

def score_table():
    return pd.DataFrame({name: evaluate(y_test, values)
                         for name, values in predictions.items()}).T

score_table().round(4)

None of these should report R² = 1.0. If one does, stop and re-read
section 1.1: that is a leak, not a breakthrough.

In [ ]:
lasso = make_pipeline(StandardScaler(), Lasso(alpha=0.1, max_iter=5000))
lasso.fit(X_train, y_train)
weights = pd.Series(lasso[-1].coef_, index=X_train.columns)
kept = weights[weights != 0].abs().sort_values(ascending=False)
print(f"Lasso kept {len(kept)} of {len(weights)} features:\n")
print(kept.head(12).round(3).to_string())

**Question.** Which features survived, and does the ranking match the
day-of-week chart in section 0?

### 2.3 Polynomial features

In [ ]:
poly = make_pipeline(StandardScaler(),
                     PolynomialFeatures(degree=2, include_bias=False),
                     Ridge(alpha=1.0))
poly.fit(X_train, y_train)
predictions["Polynomial(2) + Ridge"] = pd.Series(poly.predict(X_test),
                                                 index=X_test.index)
print(f"{X_train.shape[1]} features -> "
      f"{poly[1].transform(X_train.iloc[:1]).shape[1]} after degree-2 expansion")
print(evaluate(y_test, predictions["Polynomial(2) + Ridge"]))

## 3. Support Vector Regression

SVR fits a tube of width `epsilon` around the data and penalises only what falls
outside. `C` trades tube violations against flatness; `gamma` sets how local the
RBF kernel is.

In [ ]:
for name, estimator in (("SVR linear", SVR(kernel="linear", C=1.0, epsilon=0.1)),
                        ("SVR rbf", SVR(kernel="rbf", C=1.0, epsilon=0.1)),
                        ("SVR poly(2)", SVR(kernel="poly", degree=2, C=1.0))):
    predictions[name] = fit_predict(estimator, X_train, y_train, X_test)
score_table().round(4)

### 3.1 Tuning with a time-series split

`GridSearchCV` defaults to K-fold, which shuffles past and future together. Use
`TimeSeriesSplit`: it only ever validates on data *after* the fold it trained on.

In [ ]:
search = GridSearchCV(
    make_pipeline(StandardScaler(), SVR(kernel="rbf")),
    {"svr__C": [1, 10, 100], "svr__gamma": ["scale", 0.01, 0.1],
     "svr__epsilon": [0.01, 0.1]},
    cv=TimeSeriesSplit(n_splits=3),
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)
search.fit(X_train, y_train)
print(f"best parameters: {search.best_params_}")
print(f"best CV RMSE   : {-search.best_score_:.4f}")

predictions["SVR rbf (tuned)"] = pd.Series(search.predict(X_test),
                                           index=X_test.index)
print(evaluate(y_test, predictions["SVR rbf (tuned)"]))

## 4. Model comparison and evaluation

### 4.1 Performance table

In [ ]:
comparison = score_table().sort_values("rmse")
print(comparison.round(4).to_string())

plt.figure(figsize=(12, 5))
plt.barh(comparison.index[::-1], comparison["rmse"][::-1])
plt.xlabel("RMSE")
plt.title("Model comparison (lower is better)")
plt.tight_layout();

### 4.2 Predictions against the truth

In [ ]:
best_name = comparison.index[0]
best_prediction = predictions[best_name]

plt.figure(figsize=(20, 5))
plt.plot(y_train.iloc[-180:], color="lightgray", label="train")
plt.plot(y_test, color="black", linewidth=1, label="actual")
plt.plot(best_prediction, color="tab:red", linewidth=1, label="predicted")
plt.legend()
plt.title(f"Best by RMSE: {best_name}");

### 4.3 Residual analysis

In [ ]:
residuals = y_test - best_prediction

fig, axes = plt.subplots(1, 3, figsize=(20, 4))
axes[0].plot(residuals); axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_title("Residuals over time")
axes[1].hist(residuals, bins=30); axes[1].set_title("Residual distribution")
axes[2].scatter(y_test, best_prediction, s=6, alpha=0.5)
limits = [y_test.min(), y_test.max()]
axes[2].plot(limits, limits, "r--")
axes[2].set_xlabel("actual"); axes[2].set_ylabel("predicted")
axes[2].set_title("Predicted vs actual")
plt.tight_layout()

print(f"residual mean {residuals.mean():.3f}, std {residuals.std():.3f}")
print(f"residual autocorrelation at lag 1: {residuals.autocorr(1):.3f}")

### 4.4 Questions for analysis

1. Which model won, and by how much over naive? Was the extra complexity worth
   it?
2. Linear models against SVR — what does the gap say about how non-linear this
   series really is?
3. Which features did Lasso keep, which did it zero, and does that match the
   day-of-week plot?
4. Residual autocorrelation at lag 1 is printed above. If it is far from zero,
   what is the model still missing?
5. Compare against Holt-Winters (Practice 2) and ARIMA (Practice 4) — on effort
   as well as RMSE.
6. **The one that matters.** A leaking feature set scores R² = 1.000. Without
   the warning in section 1.1, what would have made you suspicious?

In [ ]:
# your code here — free exploration, not graded